## OpenQASM export for local measurement settings

This short example shows how to export `LocalUnitaryMeasurementSetting` and `ComputationalBasisMeasurementSetting` to OpenQASM 2.0, and how to validate the internal `u3` angle extraction on Haar-random single-qubit gates.

In [1]:
using RandomMeas
using LinearAlgebra
using Random
using Statistics

In [2]:
# Build a random local unitary measurement setting (Haar ensemble).
N = 3
ξ = siteinds("Qubit", N)
setting_local = LocalUnitaryMeasurementSetting(N; site_indices=ξ, ensemble=Haar)

LocalUnitaryMeasurementSetting(3, ITensor[ITensor ord=2
Dim 1: (dim=2|id=529|"Qubit,Site,n=1")'
Dim 2: (dim=2|id=529|"Qubit,Site,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 -0.13212339377229698 + 0.6353956745400265im  …   0.4364429672756355 - 0.6231639286010973im
  0.06039225595208801 + 0.758398655731227im      -0.2081439765140829 + 0.614703456511341im
, ITensor ord=2
Dim 1: (dim=2|id=38|"Qubit,Site,n=2")'
Dim 2: (dim=2|id=38|"Qubit,Site,n=2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
  -0.708091437042133 + 0.26398770044376285im  …  -0.6465774596733417 + 0.10418540898028583im
 -0.6505822105491953 - 0.0752316297772678im       0.7191652345057253 + 0.2321300382895412im
, ITensor ord=2
Dim 1: (dim=2|id=226|"Qubit,Site,n=3")'
Dim 2: (dim=2|id=226|"Qubit,Site,n=3")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
   0.6921692921055365 + 0.5213381297955587im   …  -0.318264566380492 + 0.3844683228486151im
 -0.24380313996789413 + 0.43550918983321696im      0.73589

In [4]:
# Convert to OpenQASM string and inspect the first lines.
qasm_local = to_OpenQASM(setting_local)
println(join(split(qasm_local, '\n'), '\n'))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
u3(1.729088023341444,-0.2844808070854094,0.4059545026936213) q[0];
u3(1.428147049734463,0.4719804654990405,-2.944498695041696) q[1];
u3(1.045136858873596,1.435598811029231,-1.524872396928059) q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];



In [5]:
# Save the OpenQASM program to disk.
path_local = "local_setting.qasm"
export_OpenQASM(setting_local, path_local)
println("Wrote: " * path_local)

Wrote: local_setting.qasm


In [6]:
# Computational-basis setting: no u3 rotations are emitted.
setting_comp = ComputationalBasisMeasurementSetting(N; site_indices=ξ)
qasm_comp = to_OpenQASM(setting_comp)
println(join(split(qasm_comp, '\n'), '\n'))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];



### Optional validation of `u3` angle extraction

`RandomMeas._u3_angles_from_unitary` is an internal helper. The check below reconstructs the `u3` matrix and compares it to the original unitary up to a global phase.

In [7]:
# OpenQASM u3 matrix convention.
u3_matrix(θ, ϕ, λ) = ComplexF64[
    cos(θ / 2) -exp(1im * λ) * sin(θ / 2);
    exp(1im * ϕ) * sin(θ / 2) exp(1im * (ϕ + λ)) * cos(θ / 2)
]

# Compare up to a global phase.
phase_invariant_error(U, V) = begin
    α = angle(sum(conj.(V) .* U))
    norm(U - exp(1im * α) * V)
end

# One fixed gate and one Haar-random gate.
for U in (ComplexF64[0 1; 1 0], begin
    Random.seed!(123)
    A = randn(ComplexF64, 2, 2)
    F = qr(A)
    Q = Matrix(F.Q)
    R = Matrix(F.R)
    d = diag(R)
    phases = map(x -> iszero(x) ? (1.0 + 0im) : x / abs(x), d)
    Q * Diagonal(phases)
end)
    θ, ϕ, λ = RandomMeas._u3_angles_from_unitary(U)
    Urec = u3_matrix(θ, ϕ, λ)
    println(phase_invariant_error(U, Urec))
end

1.3196575456672638e-16
1.2490009027033011e-15


In [8]:
# Small Monte Carlo check on Haar-random single-qubit gates.
Random.seed!(7)
n_samples = 100
errs = Float64[]
for _ in 1:n_samples
    A = randn(ComplexF64, 2, 2)
    F = qr(A)
    Q = Matrix(F.Q)
    R = Matrix(F.R)
    d = diag(R)
    phases = map(x -> iszero(x) ? (1.0 + 0im) : x / abs(x), d)
    U = Q * Diagonal(phases)

    θ, ϕ, λ = RandomMeas._u3_angles_from_unitary(U)
    Urec = u3_matrix(θ, ϕ, λ)
    push!(errs, phase_invariant_error(U, Urec))
end
println("max error = " * string(maximum(errs)))
println("mean error = " * string(mean(errs)))

max error = 1.2562320778793438e-15
mean error = 4.56929473575175e-16
